<a href="https://colab.research.google.com/github/Darwisher/Big-Data-Group_Activity/blob/main/Lab_Activity_3_Your_Data%3F_Our_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import pyspark
from pyspark.sql import SparkSession

sprk = SparkSession.builder.appName('NBA_RDD_Project').getOrCreate()
sc = spark.sparkContext

raw_rdd = sc.textFile("/content/2021-2022 NBA Player Stats - Regular.csv")
header = raw_rdd.first()

# Filter header and split by semicolon, as the CSV uses ';' as a delimiter
data_rdd = raw_rdd.filter(lambda line: line != header).map(lambda line: line.split(";"))

In [21]:

# Filter out 'TOT' (Total rows for traded players to avoid double counting)
# Team | Index 5: Games Played (G) | Index 29: Points Per Game (PTS)
clean_data = data_rdd.filter(lambda x: x[4] != 'TOT')

# Calculate Season Total for each player and map to (Team, PlayerSeasonTotal)
# Formula: (Games Played * Points Per Game)
team_season_rdd = clean_data.map(lambda x: (
    x[4],
    float(x[5]) * float(x[29])
)).partitionBy(30) # Hash Partitioning Strategy